# Lab 3 — Hive: tabelas Managed e External

## Objetivo

Este laboratório apresenta a diferença entre tabelas gerenciadas e tabelas externas.

Em uma tabela External, o catálogo armazena apenas a definição e a localização dos dados. Assim, a exclusão da tabela não deve apagar o arquivo original.

Em uma tabela Managed, os dados são copiados para o armazenamento administrado pelo banco. Em um ambiente Hive, a exclusão da tabela também pode excluir os arquivos sob responsabilidade do Hive.

Nesta rota local, o DuckDB será utilizado para representar os dois conceitos.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pasta_projeto = Path.cwd().resolve()

if not (pasta_projeto / "dados" / "raw").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "raw").exists():
            pasta_projeto = pasta_pai
            break

arquivo_clientes = (
    pasta_projeto
    / "dados"
    / "raw"
    / "sqoop_import"
    / "customers"
    / "customers_from_db.csv"
)

pasta_lab03 = (
    pasta_projeto
    / "dia2_transformacao"
    / "lab03_hive_tabelas"
)

arquivo_banco = pasta_lab03 / "bigdata_course.duckdb"

assert arquivo_clientes.exists(), (
    f"Arquivo não encontrado: {arquivo_clientes}"
)

print("Projeto:", pasta_projeto)
print("Arquivo externo:", arquivo_clientes)
print("Banco DuckDB:", arquivo_banco)

Projeto: C:\BigData\bigdata-curso-gabriel
Arquivo externo: C:\BigData\bigdata-curso-gabriel\dados\raw\sqoop_import\customers\customers_from_db.csv
Banco DuckDB: C:\BigData\bigdata-curso-gabriel\dia2_transformacao\lab03_hive_tabelas\bigdata_course.duckdb


In [2]:
conexao = duckdb.connect(str(arquivo_banco))

print("Conexão com o DuckDB realizada com sucesso.")

Conexão com o DuckDB realizada com sucesso.


In [3]:
caminho_sql_clientes = arquivo_clientes.as_posix().replace("'", "''")

conexao.execute(f"""
    CREATE OR REPLACE VIEW raw_customers AS
    SELECT *
    FROM read_csv_auto(
        '{caminho_sql_clientes}',
        header = true
    )
""")

quantidade_external = conexao.execute("""
    SELECT COUNT(*)
    FROM raw_customers
""").fetchone()[0]

print("Linhas na tabela External:", quantidade_external)

Linhas na tabela External: 9993


In [4]:
schema_external = conexao.execute("""
    DESCRIBE raw_customers
""").df()

schema_external

,column_name,column_type,null,key,default,extra
0,customer_id,BIGINT,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,cpf,VARCHAR,YES,None,None,None
3,email,VARCHAR,YES,None,None,None
4,segment,VARCHAR,YES,None,None,None
5,credit_score,BIGINT,YES,None,None,None
6,created_at,DATE,YES,None,None,None


In [5]:
conexao.execute("""
    DROP TABLE IF EXISTS managed_customers
""")

conexao.execute("""
    CREATE TABLE managed_customers AS
    SELECT *
    FROM raw_customers
""")

quantidade_managed = conexao.execute("""
    SELECT COUNT(*)
    FROM managed_customers
""").fetchone()[0]

comparacao_tabelas = pd.DataFrame({
    "Tipo": ["External", "Managed"],
    "Objeto": ["raw_customers", "managed_customers"],
    "Linhas": [quantidade_external, quantidade_managed],
    "Onde estão os dados": [
        "CSV da camada Raw",
        "Dentro do banco DuckDB"
    ]
})

comparacao_tabelas

,Tipo,Objeto,Linhas,Onde estão os dados
0,External,raw_customers,9993,CSV da camada Raw
1,Managed,managed_customers,9993,Dentro do banco DuckDB


In [6]:
conexao.execute("""
    DELETE FROM managed_customers
    WHERE customer_id <= 10
""")

quantidade_external_apos_delete = conexao.execute("""
    SELECT COUNT(*)
    FROM raw_customers
""").fetchone()[0]

quantidade_managed_apos_delete = conexao.execute("""
    SELECT COUNT(*)
    FROM managed_customers
""").fetchone()[0]

teste_independencia = pd.DataFrame({
    "Objeto": [
        "External: raw_customers",
        "Managed: managed_customers"
    ],
    "Linhas após o DELETE": [
        quantidade_external_apos_delete,
        quantidade_managed_apos_delete
    ]
})

teste_independencia

,Objeto,Linhas após o DELETE
0,External: raw_customers,9993
1,Managed: managed_customers,9983


In [7]:
arquivo_existia_antes = arquivo_clientes.exists()

conexao.execute("""
    DROP VIEW raw_customers
""")

conexao.execute("""
    DROP TABLE managed_customers
""")

arquivo_existe_depois = arquivo_clientes.exists()

resultado_drop = pd.DataFrame({
    "Verificação": [
        "CSV existia antes do DROP",
        "CSV existe depois do DROP"
    ],
    "Resultado": [
        arquivo_existia_antes,
        arquivo_existe_depois
    ]
})

resultado_drop

,Verificação,Resultado
0,CSV existia antes do DROP,True
1,CSV existe depois do DROP,True


In [8]:
objetos_apos_drop = conexao.execute("""
    SELECT
        table_name,
        table_type
    FROM information_schema.tables
    WHERE table_schema = 'main'
      AND table_name IN (
          'raw_customers',
          'managed_customers'
      )
    ORDER BY table_name
""").df()

if objetos_apos_drop.empty:
    print("Os dois objetos foram removidos do catálogo.")
else:
    display(objetos_apos_drop)

Os dois objetos foram removidos do catálogo.


In [9]:
conexao.execute(f"""
    CREATE OR REPLACE VIEW raw_customers AS
    SELECT *
    FROM read_csv_auto(
        '{caminho_sql_clientes}',
        header = true
    )
""")

validacao_final = conexao.execute("""
    SELECT COUNT(*) AS quantidade
    FROM raw_customers
""").df()

validacao_final

,quantidade
0,9993


In [10]:
conexao.close()

with duckdb.connect(str(arquivo_banco)) as teste_conexao:
    quantidade_persistida = teste_conexao.execute("""
        SELECT COUNT(*)
        FROM raw_customers
    """).fetchone()[0]

print("Linhas após reabrir o banco:", quantidade_persistida)

Linhas após reabrir o banco: 9993


## Conclusão

A tabela `raw_customers` foi criada como uma visão sobre o CSV armazenado na camada Raw, representando uma tabela External. Nessa estrutura, o DuckDB mantém a definição da visão, mas os dados permanecem no arquivo externo.

A tabela `managed_customers`, por sua vez, recebeu uma cópia dos dados dentro do banco DuckDB. A exclusão de dez registros da tabela Managed não afetou a tabela External, que continuou apresentando os 9.993 clientes originais.

Após a exclusão dos dois objetos do catálogo, o CSV permaneceu no sistema de arquivos. Isso demonstra que o DuckDB não exclui o arquivo de origem quando uma visão ou tabela é removida.

No Hive, a distinção exige cuidado adicional: o comando `DROP TABLE` aplicado a uma tabela Managed pode excluir tanto a definição quanto os arquivos administrados pelo Hive. Em uma tabela External, normalmente apenas a definição do catálogo é removida, preservando-se os dados externos.